# 09 — Generazione delle tracce SAM2 sulle 7 clip (input di E3)

Esegue la **pipeline originale** (YOLO11s Roboflow → SAM2 → keypoint del campo → omografia) su tutte le clip del manifest e produce, per ogni clip, gli stessi file del pilot `clip_01`:

- `<clip>_tracks.csv` → `frame, track_id, mask_area_px, x1, y1, x2, y2, foot_x, foot_y`
- `<clip>_positions_2d.csv` → `frame, track_id, x_ft, y_ft, status`
- `<clip>_homographies_image_to_court_ft.json` → `{frame: H}` (pixel → campo in piedi)
- `<clip>_tracking.mp4` → video con gli ID, per il controllo visivo

Due varianti (`MODES`):
- `f0`: baseline originale, SAM2 inizializzato solo con le detection del frame 0;
- `redet`: **miglioria** — ogni `REDETECT_EVERY` frame il detector cerca giocatori non coperti da nessuna maschera; se confermati in due controlli consecutivi vengono aggiunti (risolve il portatore di palla mancante della clip_01).

Alla fine scrive `annotations/manifest_sam2_<mode>.csv` con i percorsi compilati → li usa il notebook 12.
Runtime: **Colab GPU T4**. Serve il Secret `ROBOFLOW_API_KEY` (icona chiave a sinistra) per detector e keypoint.

In [ ]:
%pip install -q "inference" "supervision==0.30.9" "git+https://github.com/roboflow/sports.git@feat/basketball"

In [ ]:
# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # riduce la frammentazione della memoria GPU
import json, time, subprocess, importlib
import numpy as np, pandas as pd, cv2, torch
import supervision as sv
from tqdm.auto import tqdm

MODES = ['f0', 'redet']
CLIPS = None                 # None = tutte le clip del manifest; oppure es. ['clip_01']
CONFIDENCE, NMS_IOU = 0.4, 0.70
REDETECT_EVERY, CONF_NEW = 5, 0.5
PLAYER_MODEL_ID = 'basketball-player-detection-3-ycjdo/4'
COURT_MODEL_ID = 'basketball-court-detection-2/14'
PLAYER_CLASSES = {'player', 'player-in-possession', 'player-jump-shot', 'player-layup-dunk', 'player-shot-block'}
MANIFEST = ROOT / 'annotations' / 'manifest.csv'
OUT_BASE = ROOT / 'outputs'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NESSUNA')

## 1. Modelli: detector YOLO11s (Roboflow), keypoint del campo, SAM2

In [ ]:
from inference import get_model
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = os.environ.get('ROBOFLOW_API_KEY') or __import__('getpass').getpass('API key Roboflow: ')
player_model = get_model(model_id=PLAYER_MODEL_ID, api_key=API_KEY)
court_model = get_model(model_id=COURT_MODEL_ID, api_key=API_KEY)

SAM2_DIR = FAST / 'segment-anything-2-real-time'
SAM2_COMMIT = 'e3623cda3f748df999f76db663f08da7a884a820'
SAM2_CKPT = ROOT / 'models' / 'sam2.1_hiera_large.pt'
if not SAM2_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Gy920/segment-anything-2-real-time.git', str(SAM2_DIR)], check=True)
    subprocess.run(['git', '-C', str(SAM2_DIR), 'checkout', SAM2_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SAM2_DIR)], check=True)
if not SAM2_CKPT.is_file():
    SAM2_CKPT.parent.mkdir(exist_ok=True)
    subprocess.run(['wget', '-q', '-O', str(SAM2_CKPT),
                    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'], check=True)
sys.path.insert(0, str(SAM2_DIR))
for m in [m for m in sys.modules if m == 'sam2' or m.startswith('sam2.')]: del sys.modules[m]
importlib.invalidate_caches()
from sam2.build_sam import build_sam2_camera_predictor
cwd = os.getcwd(); os.chdir(SAM2_DIR)
predictor = build_sam2_camera_predictor('configs/sam2.1/sam2.1_hiera_l.yaml', str(SAM2_CKPT), device='cuda')
os.chdir(cwd)
print('Modelli pronti.')

## 2. Funzioni (stesse regole della baseline originale)

In [ ]:
def nms(boxes, scores, thr=NMS_IOU):
    order, keep = np.argsort(-scores), []
    while len(order):
        i = order[0]; keep.append(i); rest = order[1:]
        if not len(rest): break
        tl = np.maximum(boxes[i, :2], boxes[rest, :2]); br = np.minimum(boxes[i, 2:], boxes[rest, 2:])
        inter = np.prod(np.clip(br - tl, 0, None), axis=1)
        a = lambda b: np.prod(np.clip(b[..., 2:] - b[..., :2], 0, None), axis=-1)
        order = rest[inter / np.maximum(a(boxes[i]) + a(boxes[rest]) - inter, 1e-8) <= thr]
    return np.array(keep, int)

def detect_players(frame, conf):
    r = player_model.infer(frame, confidence=conf, iou_threshold=NMS_IOU)[0]
    p = [x for x in r.predictions if x.class_name in PLAYER_CLASSES]
    if not p: return np.zeros((0, 4), np.float32)
    b = np.array([[x.x - x.width/2, x.y - x.height/2, x.x + x.width/2, x.y + x.height/2] for x in p], np.float32)
    s = np.array([x.confidence for x in p], np.float32)
    return b[nms(b, s)]

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

def mask_rows(object_ids, logits, fi):
    # Stesso formato e stessa pulizia delle maschere del notebook 01 originale.
    masks = (logits > 0).detach().cpu().numpy()
    if masks.ndim == 4: masks = masks[:, 0]
    rows, boxes = [], {}
    for oid, m in zip(object_ids, masks):
        oid = int(oid); m = m.astype(bool)
        if m.any(): m = sv.filter_segments_by_distance(m, relative_distance=0.03, mode='edge').astype(bool)
        ys, xs = np.where(m)
        if len(xs) == 0:
            rows.append([fi, oid, 0, '', '', '', '', '', '']); continue
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        rows.append([fi, oid, int(m.sum()), x1, y1, x2, y2, (x1 + x2) / 2, float(y2)])
        boxes[oid] = np.array([x1, y1, x2, y2], np.float32)
    return rows, boxes

def sam2_init(rgb, boxes_by_id):
    if hasattr(predictor, 'reset_state'):
        try: predictor.reset_state()
        except Exception: pass
    predictor.frame_idx = 0
    predictor.load_first_frame(rgb)
    ids, logits = [], None
    for oid, b in boxes_by_id.items():
        _, ids, logits = predictor.add_new_prompt(frame_idx=0, obj_id=int(oid), bbox=np.asarray(b, np.float32).reshape(1, 4))
    return ids, logits

def prune_memory(keep=16):
    # Il predictor real-time conserva gli output di TUTTI i frame: teniamo solo gli ultimi `keep`.
    # SAM2 usa al massimo 7 frame di memoria + 15 puntatori agli oggetti, quindi il risultato non cambia.
    st = getattr(predictor, 'condition_state', None)
    if not isinstance(st, dict): return
    lim = predictor.frame_idx - keep
    for od in [st.get('output_dict', {})] + list(st.get('output_dict_per_obj', {}).values()):
        d = od.get('non_cond_frame_outputs', {}) if isinstance(od, dict) else {}
        for k in [k for k in list(d) if isinstance(k, int) and k < lim]:
            del d[k]

def track_clip(frames, mode):
    b0 = detect_players(frames[0], CONFIDENCE)
    if len(b0) == 0: raise RuntimeError('Nessun giocatore al frame 0')
    boxes = {i + 1: b for i, b in enumerate(b0)}
    next_id, pending, missing, rows, added = len(boxes) + 1, np.zeros((0, 4)), {}, [], []
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    with torch.inference_mode(), torch.autocast('cuda', dtype=dtype):
        for fi, f in enumerate(tqdm(frames, leave=False)):
            rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            ids, logits = sam2_init(rgb, boxes) if fi == 0 else predictor.track(rgb)
            prune_memory()
            r, cur = mask_rows(ids, logits, fi); rows += r
            if mode == 'redet' and fi > 0 and fi % REDETECT_EVERY == 0:
                for oid in [int(i) for i in ids]:
                    missing[oid] = 0 if oid in cur else missing.get(oid, 0) + 1
                d = detect_players(f, CONF_NEW)
                cb = np.array(list(cur.values())).reshape(-1, 4)
                cand = d[box_iou(d, cb).max(axis=1, initial=0) < 0.3] if len(d) else np.zeros((0, 4))
                conf_ = cand[box_iou(cand, pending).max(axis=1, initial=0) > 0.5] if len(cand) else cand
                pending = cand
                if len(conf_):
                    alive = {o: b for o, b in cur.items() if missing.get(o, 0) <= 2}
                    for b in conf_:
                        alive[next_id] = b; added.append((fi, next_id)); next_id += 1
                    sam2_init(rgb, alive); pending = np.zeros((0, 4))
    torch.cuda.empty_cache()
    t = pd.DataFrame(rows, columns=['frame', 'track_id', 'mask_area_px', 'x1', 'y1', 'x2', 'y2', 'foot_x', 'foot_y'])
    return t, added

def court_geometry(frames, court_vertices):
    hs, quality = {}, []
    for fi, f in enumerate(tqdm(frames, leave=False, desc='campo')):
        res = court_model.infer(f, confidence=0.3)[0]
        n_pts = n_in = 0; err = ''; status = 'no_court'
        if res.predictions:
            pr = max(res.predictions, key=lambda p: p.confidence)
            pts = [p for p in pr.keypoints if p.confidence >= 0.5 and 0 <= p.class_id < len(court_vertices)]
            n_pts = len(pts); status = 'too_few_points'
            if n_pts >= 6:
                src = court_vertices[[p.class_id for p in pts]]; dst = np.float32([[p.x, p.y] for p in pts])
                fwd, mask = cv2.findHomography(src, dst, cv2.RANSAC, 8.0); status = 'fit_failed'
                if fwd is not None and mask is not None:
                    acc = mask.ravel().astype(bool); n_in = int(acc.sum()); status = 'too_few_inliers'
                    if n_in >= 6:
                        try: inv = np.linalg.inv(fwd)
                        except np.linalg.LinAlgError: inv = None
                        if inv is not None and np.isfinite(inv).all():
                            proj = cv2.perspectiveTransform(src.reshape(-1, 1, 2), fwd).reshape(-1, 2)
                            err = float(np.median(np.linalg.norm(proj - dst, axis=1)[acc]))
                            hs[str(fi)] = inv.tolist(); status = 'fit_available'
        quality.append([fi, n_pts, n_in, err, status])
    return hs, pd.DataFrame(quality, columns=['frame', 'keypoints', 'inliers', 'median_fit_error_px', 'status'])

def project(tracks, hs, court_cfg):
    out = []
    for r in tracks.itertuples():
        x = y = ''
        H = hs.get(str(int(r.frame)))
        if int(r.mask_area_px) == 0: st = 'missing_mask'
        elif H is None: st = 'missing_geometry'
        else:
            v = np.asarray(H) @ np.array([float(r.foot_x), float(r.foot_y), 1.0])
            if not np.isfinite(v).all() or abs(v[2]) < 1e-10: st = 'invalid_projection'
            else:
                x, y = v[0] / v[2], v[1] / v[2]
                st = 'projected' if (0 <= x <= court_cfg.court_length and 0 <= y <= court_cfg.court_width) else 'outside_court'
        out.append([r.frame, r.track_id, x, y, st])
    return pd.DataFrame(out, columns=['frame', 'track_id', 'x_ft', 'y_ft', 'status'])

def render(frames, tracks, path, fps):
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    by = dict(tuple(tracks[tracks.mask_area_px > 0].groupby('frame')))
    for fi, f in enumerate(frames):
        img = f.copy()
        for r in by.get(fi, pd.DataFrame()).itertuples():
            c = (0, 255, 255) if r.track_id % 2 else (255, 200, 0)
            cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), c, 2)
            cv2.putText(img, f'ID {r.track_id}', (int(r.x1), max(15, int(r.y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, .5, c, 2)
        cv2.putText(img, f'frame {fi}', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)
        vw.write(img)
    vw.release()

## 3. Esecuzione su tutte le clip

In [ ]:
from sports import MeasurementUnit
from sports.basketball import CourtConfiguration, League
court_cfg = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
V = np.asarray(court_cfg.vertices, dtype=np.float32)

man = pd.read_csv(MANIFEST)
todo = man if CLIPS is None else man[man.clip_id.isin(CLIPS)]
log = []
for r in todo.itertuples():
    cap = cv2.VideoCapture(str(ROOT / r.video)); frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    print(f'{r.clip_id}: {len(frames)} frame')
    geo_dir = OUT_BASE / 'geometry' / r.clip_id; geo_dir.mkdir(parents=True, exist_ok=True)
    hfile = geo_dir / f'{r.clip_id}_homographies_image_to_court_ft.json'
    if hfile.is_file():
        hs = json.loads(hfile.read_text())
    else:
        hs, q = court_geometry(frames, V)
        hfile.write_text(json.dumps(hs)); q.to_csv(geo_dir / f'{r.clip_id}_geometry_quality.csv', index=False)
    for mode in MODES:
        d = OUT_BASE / f'sam2_{mode}' / r.clip_id; d.mkdir(parents=True, exist_ok=True)
        tfile = d / f'{r.clip_id}_tracks.csv'
        if tfile.is_file():
            print(f'  [{mode}] già presente, salto'); continue
        t0 = time.time()
        tracks, added = track_clip(frames, mode)
        tracks.to_csv(tfile, index=False)
        project(tracks, hs, court_cfg).to_csv(d / f'{r.clip_id}_positions_2d.csv', index=False)
        (d / f'{r.clip_id}_homographies_image_to_court_ft.json').write_text(json.dumps(hs))
        render(frames, tracks, d / f'{r.clip_id}_tracking.mp4', r.fps)
        log.append({'clip_id': r.clip_id, 'mode': mode, 'seconds': round(time.time() - t0, 1),
                    'ids': tracks.track_id.nunique(), 'ids_added': len(added),
                    'frames_with_H': len(hs), 'frames': len(frames)})
        print(f'  [{mode}] ID={tracks.track_id.nunique()} aggiunti={len(added)} H={len(hs)}/{len(frames)}  {time.time()-t0:.0f}s')
pd.DataFrame(log)

## 4. Manifest compilati per il notebook 12

In [ ]:
for mode in MODES:
    m = man.copy()
    for col, suffix in [('tracks_csv', '_tracks.csv'), ('positions_csv', '_positions_2d.csv'),
                        ('homographies_json', '_homographies_image_to_court_ft.json')]:
        m[col] = [f'outputs/sam2_{mode}/{c}/{c}{suffix}' if (OUT_BASE / f'sam2_{mode}' / c / f'{c}{suffix}').is_file() else ''
                  for c in m.clip_id]
    out = ROOT / 'annotations' / f'manifest_sam2_{mode}.csv'
    m.to_csv(out, index=False)
    print(out.name, '-> clip complete:', int((m.tracks_csv != '').sum()), '/', len(m))